# Two-Tower Retrieval Model
**Stage: Candidate Generation (Retrieval)**

Based on the Databricks reference — ported to run free on **Google Colab T4** or **Kaggle P100**.
No Spark, no Databricks, no cloud storage required.

## Architecture
```
user_id ─► UserTower (Embedding → LayerNorm → MLP) ─► L2-norm ─►┐
                                                                    ├─► dot product ─► logit
item_id ─► ItemTower (Embedding → LayerNorm → MLP) ─► L2-norm ─►┘
```
**Used by:** YouTube DNN, Pinterest PinSage, Airbnb, Meta DLRM retrieval stage

## Stack
| Databricks Original | This Notebook |
|---|---|
| TorchDistributor (PySpark) | Standard PyTorch single/multi-GPU |
| Mosaic StreamingDataset (S3) | torch DataLoader + pandas |
| dbutils / spark.sql | Removed — pure Python |
| MLflow (Databricks workspace) | mlflow 3.11 (local tracking) |
| Delta Lake | MovieLens 1M CSV (auto-downloaded) |
| TorchRec | Standard nn.Embedding |

## Free GPU Options
- **Google Colab**: Runtime → Change runtime type → T4 GPU
- **Kaggle**: Settings → Accelerator → GPU (30 hrs/week free)
- **Paperspace Gradient**: Free M4000 tier


In [ ]:
# ── 0. Install (run once, then restart runtime if on Colab) ───────────────
import sys
pkgs = [
    'torch==2.11.0',
    'mlflow==3.11.1',
    'scikit-learn==1.8.0',
    'pandas==2.2.3',
    'numpy==1.26.4',
    'matplotlib',
    'tqdm',
]
!{sys.executable} -m pip install --quiet {' '.join(pkgs)}
print('Done. Now go to Runtime → Restart session, then run from Cell 1.')

In [ ]:
import subprocess
import os
import shutil

repo_url = "https://github.com/mutellipiqbal/two-tower-recommender"
repo_dir = "two-tower-recommender"

# Ensure clean state: remove the directory if it exists
if os.path.exists(repo_dir):
    print(f"Removing existing repository directory '{repo_dir}'...")
    shutil.rmtree(repo_dir)
    print(f"✓ Repository directory '{repo_dir}' removed.")

print(f"Cloning {repo_url}...")
try:
    subprocess.run(['git', 'clone', repo_url], check=True)
    if os.path.exists(repo_dir):
        print(f"✓ Repository '{repo_dir}' cloned successfully.")
    else:
        print(f"✗ Repository '{repo_dir}' was not created despite git reporting success.")
except subprocess.CalledProcessError as e:
    print(f"✗ Error cloning repository: {e}")
except FileNotFoundError:
    print("✗ 'git' command not found. Ensure Git is installed and in your PATH.")

In [ ]:
import requests

try:
    response = requests.get('http://www.google.com', timeout=5)
    response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
    print('✓ Successfully connected to the internet.')
except requests.exceptions.ConnectionError:
    print('✗ Connection Error: Could not reach the internet. Check proxy settings or network access.')
except requests.exceptions.Timeout:
    print('✗ Timeout Error: Request took too long. Network might be slow or blocked.')
except requests.exceptions.RequestException as e:
    print(f'✗ An unexpected error occurred: {e}')

In [ ]:
!ls -R two-tower-recommender

In [ ]:
import os, sys

# Find and move into the project folder
for path in [
    "/content/two-tower-recommender",   # if you cloned it
    "/content",                          # if you opened notebook directly
    ".",                                 # current dir fallback
]:
    if os.path.exists(os.path.join(path, "src")):
        os.chdir(path)
        sys.path.insert(0, path)
        print(f"✓ Working directory set to: {path}")
        break
else:
    print("✗ src/ folder not found. See fix below.")

# Verify
print("Files here:", os.listdir("."))

In [ ]:
# ── 1. Imports & Config ───────────────────────────────────────────────────
import sys, os
sys.path.insert(0, '.')   # so 'src' is importable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import mlflow
from tqdm.auto import tqdm

from src.model   import TwoTowerModel
from src.dataset import download_movielens, load_and_encode, build_interaction_df, split_data, make_loaders
from src.trainer import train, evaluate
from src.evaluate import evaluate_retrieval

# ── Config ────────────────────────────────────────────────────────────────
CFG = {
    'embedding_dim':      64,
    'hidden_dims':        [256, 128, 64],
    'dropout':            0.1,
    'batch_size':         4096,
    'lr':                 3e-4,
    'weight_decay':       1e-5,
    'epochs':             15,
    'pos_threshold':      4.0,     # ratings >= 4 = positive
    'neg_ratio':          4,       # negatives per positive
    'in_batch_negatives': True,    # industry-standard training objective
    'seed':               42,
    'mlflow_experiment':  'two_tower_retrieval',
}

torch.manual_seed(CFG['seed'])
np.random.seed(CFG['seed'])
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU   : {torch.cuda.get_device_name(0)}')

## 2. Data — MovieLens 1M

In [ ]:
download_movielens()
ratings, user2idx, item2idx, NUM_USERS, NUM_ITEMS = load_and_encode(pos_threshold=CFG['pos_threshold'])
df = build_interaction_df(ratings, CFG['pos_threshold'], CFG['neg_ratio'], CFG['seed'], NUM_ITEMS)
train_df, val_df, test_df = split_data(df, seed=CFG['seed'])
train_loader, val_loader, test_loader = make_loaders(train_df, val_df, test_df, CFG['batch_size'])
print(f'Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}')

## 3. Model

In [ ]:
model = TwoTowerModel(
    num_users=NUM_USERS,
    num_items=NUM_ITEMS,
    embedding_dim=CFG['embedding_dim'],
    hidden_dims=CFG['hidden_dims'],
    dropout=CFG['dropout'],
).to(DEVICE)

# Optional: torch.compile for ~20% speedup on GPU (PyTorch 2.x)
if DEVICE.type == 'cuda':
    model = torch.compile(model)

total_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {total_params:,}')

## 4. Train (MLflow 3.x tracked)

In [ ]:
model = train(
    model, train_loader, val_loader,
    cfg=CFG, device=DEVICE, run_name='two_tower_v1'
)

## 5. Evaluate — AUC on Test Set

In [ ]:
loss_fn = nn.BCEWithLogitsLoss()
test_metrics = evaluate(model, test_loader, loss_fn, DEVICE)
print(f'Test AUC : {test_metrics["auc"]:.4f}')
print(f'Test Loss: {test_metrics["loss"]:.4f}')

## 6. Retrieval Evaluation — Recall@K & NDCG@K via FAISS
This is how two-tower models are actually evaluated in production:
build a FAISS index over all item embeddings, then retrieve top-K for each user.

In [ ]:
import sys
import importlib

# Attempt to install faiss-cpu if it's not already installed and available
try:
    import faiss
    print("FAISS already installed.")
except ImportError:
    print("FAISS not found. Installing faiss-cpu...")
    !{sys.executable} -m pip install --quiet faiss-cpu
    try:
        import faiss
        print("✓ faiss-cpu installed successfully.")
    except ImportError:
        print("""✗ Failed to install faiss-cpu. Please check your internet connection or try installing manually.
           You might need to restart the runtime and re-run cells from the beginning.""")

# Re-import src.evaluate to refresh the FAISS_AVAILABLE flag
# This is necessary because src.evaluate might have been imported before faiss was available.
if 'src.evaluate' in sys.modules:
    importlib.reload(sys.modules['src.evaluate'])
else:
    # If for some reason it wasn't imported (unlikely here), import it now.
    import src.evaluate

# Ensure evaluate_retrieval is the updated one from the potentially reloaded module
from src.evaluate import evaluate_retrieval

retrieval_metrics = evaluate_retrieval(
    model=model,
    test_df=test_df,
    num_items=NUM_ITEMS,
    device=DEVICE,
    k_values=[10, 20, 50],
    n_users=1000,
)
print('\nRetrieval Metrics (FAISS ANN):')
for k, v in retrieval_metrics.items():
    print(f'  {k:<15}: {v:.4f}')

## 7. Inspect MLflow Runs
Run `mlflow ui` in terminal (or in Colab below) to open the experiment dashboard.

In [ ]:
# View all runs programmatically
import mlflow
runs = mlflow.search_runs(experiment_names=['two_tower_retrieval'])
print(runs[['run_id','metrics.val_auc','metrics.best_val_auc','params.epochs']].head())

## 8. Save for Production
In production the user tower goes to a real-time serving endpoint;
the item tower runs offline to populate a vector store (e.g., Vertex Matching Engine, Pinecone, Milvus).

In [ ]:
# Export item embeddings → can be loaded into any ANN/vector store
from src.evaluate import compute_all_item_embeddings
import numpy as np

item_embs = compute_all_item_embeddings(model, NUM_ITEMS, DEVICE)
np.save('item_embeddings.npy', item_embs)
print(f'Saved item_embeddings.npy  shape={item_embs.shape}')

# Save model state dict
torch.save(model.state_dict(), 'two_tower_final.pt')
print('Saved two_tower_final.pt')

---
**Next step:** Feed the top-100 retrieved candidates into the **DLRM re-ranker** (`dlrm-recommender` project).